# Week 9: Surrogates, Estimation, and Climate Economics

**Deep Learning for Macroeconomics** — Honours, The University of Edinburgh  
Instructor: Juan Zurita · [juan.zurita@ed.ac.uk](mailto:juan.zurita@ed.ac.uk)

---

## Learning objectives

By the end of this notebook you will be able to:

1. Build a **surrogate model** (neural network or Gaussian process) that replaces an expensive simulation
2. Use surrogates for **simulated method of moments (SMM)** estimation
3. Understand the **DICE integrated assessment model** for climate economics
4. Apply deep learning to solve a stochastic climate-economy model

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch {torch.__version__} on {device}")

## 1. Surrogate models — fast approximations of slow simulations

Some economic models take hours to solve once. Estimating them (which requires solving thousands of times) is infeasible without a shortcut.

**Surrogate models** replace the expensive simulation with a fast, differentiable approximation:

1. Solve the model at $N$ parameter vectors $\{\theta_1, \ldots, \theta_N\}$
2. Record the moments $m(\theta_i)$ for each solve
3. Train a neural network or GP: $\hat{m}(\theta) \approx m(\theta)$
4. Use $\hat{m}$ in the objective function: $\min_\theta (\hat{m}(\theta) - m^{\text{data}})^2$

## 2. Gaussian processes — uncertainty-aware surrogates

A **Gaussian process** (GP) provides not just a prediction but an uncertainty estimate. This is valuable for:

- **Active learning:** sample where the GP is most uncertain
- **Bayesian optimisation:** efficiently search for the best-fitting parameters
- **Credible intervals:** know when the surrogate's prediction is reliable

In [ ]:
# Simple Gaussian process regression

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel

# Simulate a model moment as a function of a parameter
np.random.seed(42)
theta_train = np.sort(np.random.uniform(0, 5, 15)).reshape(-1, 1)
# Pretend this is an expensive model solve
moment_train = np.sin(theta_train) + 0.1 * np.random.randn(*theta_train.shape)

# Fit GP
kernel = ConstantKernel() * RBF(length_scale=1.0)
gp = GaussianProcessRegressor(kernel=kernel, alpha=0.01)
gp.fit(theta_train, moment_train)

# Predict
theta_test = np.linspace(0, 5, 200).reshape(-1, 1)
mu, sigma = gp.predict(theta_test, return_std=True)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(theta_test, np.sin(theta_test), 'k--', label='True moment function', alpha=0.5)
ax.plot(theta_test, mu, color='#7a2318', linewidth=2, label='GP mean')
ax.fill_between(theta_test.flatten(), mu.flatten() - 2*sigma, mu.flatten() + 2*sigma,
                alpha=0.2, color='#7a2318', label='95% CI')
ax.scatter(theta_train, moment_train, c='black', s=50, zorder=5, label='Training solves')
ax.set_xlabel('Parameter θ', fontsize=11)
ax.set_ylabel('Model moment m(θ)', fontsize=11)
ax.set_title('Gaussian Process Surrogate', fontsize=12)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. The DICE model — climate meets economics

The **Dynamic Integrated model of Climate and the Economy** (Nordhaus, 2017) couples:
- An economic growth model (Ramsey-type)
- A carbon-cycle model
- A climate/temperature model
- A damage function linking temperature to GDP

This is a natural application for DEQNs because the state space is high-dimensional (capital, carbon concentrations, temperature layers, technology) and the model is non-stationary.

## 4. Structural estimation with surrogates

**Simulated Method of Moments (SMM):**

1. Choose target moments from data: $m^{\text{data}}$
2. For a given $\theta$, solve the model and simulate to get $m(\theta)$
3. Minimise $[m(\theta) - m^{\text{data}}]' W [m(\theta) - m^{\text{data}}]$

With a surrogate, step 2 is instant — enabling gradient-based optimisation over $\theta$.

---

## Exercises

**Exercise 1:** Build a GP surrogate for the Brock–Mirman model's steady-state capital as a function of $\alpha$ and $\beta$. How many training points do you need for 1% accuracy?

**Exercise 2:** Implement a simplified DICE model (2 state variables: capital and carbon concentration). Solve it with a DEQN and plot the optimal carbon tax path.

---

*Next week: Frontiers and Course Synthesis — when to use which method.*